# Reading tree-ring data with dplPy

This notebook contains a short tour of the functionality of `dplpy.readers()`. dplPy reads Tucson/ITRDB `.rwl` files (and .csv file) into a pandas `DataFrame` with one column per measurement series indexed by year.  `readers.py` has been substantially improved to be able to succeed despite the many quirks and format errors found in real ITRDB files. As of version v0.3.0 of dplPy, we have validated its behavior against dplR 1.7.9 using the whole ITRDB.

This notebook uses the sample files in `../tests/data/rwl/` in the dplPy repository.

In [ ]:
import io, os, shutil, tempfile, contextlib, warnings
import dplpy as dpl

DATA = "../tests/data/rwl/"

def quiet(fn, *args, **kwargs):
    """Run a reader while hiding its progress printout, to keep the demo tidy."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

## 1. The basics

No arguments needed — pass a path and get back a year-indexed `DataFrame`.

In [ ]:
data = dpl.readers(DATA + "ca533.rwl")
data.iloc[:5, :5]

## 2. Headers are detected automatically

Most ITRDB files begin with a 3-line metadata header. dplPy finds where the data
actually starts — no need to pass `header=True` — and tells you how many header
lines it skipped (handy for catching a rare mis-detection).

In [ ]:
th = quiet(dpl.readers, DATA + "th001.rwl")
print("header lines skipped:", th.attrs["dplpy_header_lines_skipped"])
th.iloc[:3, :4]

## 3. Header metadata

dplPy extracts site/sample metadata from the header. It rides along on
`df.attrs["dplpy_metadata"]`, or you can pull it directly (and cheaply, reading
only the header) with `dpl.metadata()`.

In [ ]:
meta = dpl.metadata(DATA + "tx042.rwl")
meta

In [ ]:
print(meta["site_id"], "|", meta["species_code"], "-", meta["species_name"],
      "|", meta["country_region"])
print("lat/lon:", meta["latitude"], meta["longitude"],
      "| hemisphere_verified:", meta["hemisphere_verified"])

The coordinate **sign** is cross-checked against the standardized country/state
in the header (e.g. a US state forces West longitude). When the region isn't a
recognized standardized name, the sign is left as-decoded and
`hemisphere_verified` is `False`, so you know it wasn't confirmed.

## 4. Flexible about the file suffix

A Tucson file needn't end in `.rwl`. For an unrecognized suffix dplPy sniffs the
content; you can also force it with `format=`.

In [ ]:
tmp = tempfile.mkdtemp()
alt = os.path.join(tmp, "mydata.txt")            # a Tucson file with a .txt suffix
shutil.copy(DATA + "ca533.rwl", alt)
print("read a .txt by content sniffing:", quiet(dpl.readers, alt).shape)
print("or force it:", quiet(dpl.readers, alt, format="tucson").shape)

## 5. Robust to messy files — strict mode

Real archives contain malformed files. By default (`on_error="raise"`) dplPy
refuses them with a specific, actionable message rather than silently corrupting
data.

In [ ]:
for f in ["akfirmc.rwl", "viet001.rwl", "kyrg014.rwl"]:
    try:
        quiet(dpl.readers, DATA + f)
    except ValueError as e:
        detail = [ln.strip() for ln in str(e).splitlines() if ln.strip()]
        print(f"{f}:  {detail[1] if len(detail) > 1 else detail[0]}\n")

## 6. Salvage mode — warn and continue

For batch processing a whole collection, `on_error="warn"` recovers as much as
possible instead of failing: it drops an unusable series (self-overlap or a
mixed-precision series), or renames a genuinely duplicated series ID, and records
every action on `df.attrs["dplpy_salvage"]`.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d = quiet(dpl.readers, DATA + "kyrg014.rwl", on_error="warn")
print("kyrg014 salvaged ->", d.shape)
d.attrs["dplpy_salvage"]

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d2 = quiet(dpl.readers, DATA + "viet001.rwl", on_error="warn")
print("duplicate ID kept as two series:",
      [c for c in d2.columns if c.startswith("BDF02A")])
d2.attrs["dplpy_salvage"]

## 7. Tricky real-world cases

**Mixed measurement precision within one file** (here TMS* series measured at
0.001 mm and TWM* at 0.01 mm) and **non-ASCII series IDs** are both handled.

In [ ]:
tms = quiet(dpl.readers, DATA + "TMScombined.rwl")
tms[["TMS01A", "TWM01a"]].dropna().head()

In [ ]:
ru = quiet(dpl.readers, DATA + "russ301.rwl")
[c for c in ru.columns if any(ord(ch) > 127 for ch in c)][:6]

## 8. Reading straight from a URL

`readers()` accepts an http(s) URL — for example a file from the NOAA/ITRDB
archive. (This cell needs network access; it degrades gracefully if offline.)

In [ ]:
url = ("https://www.ncei.noaa.gov/pub/data/paleo/treering/"
       "measurements/northamerica/usa/ak132x.rwl")
try:
    ak = quiet(dpl.readers, url)
    print("read from URL ->", ak.shape)
except Exception as e:
    print("(network unavailable here:", type(e).__name__, "-- works on a networked machine)")

---
That's the tour: automatic headers, metadata with coordinate correction, flexible
formats and URLs, and a strict/salvage choice for handling the messy realities of
the ITRDB. See the docstring of `dpl.readers` for the full option list.